# From Qubits to Chemistry: A Qrisp Tutorial

In this tutorial we will take you on a journey from the very basics of the Qrisp quantum programming language all the way to simulating real molecular systems and estimating the resources of large-scale quantum algorithms. We will cover:

1. **Qrisp Basics** - QuantumVariables, gates, functions, and measurements
2. **Hamiltonian Simulation of $H_2$** - Loading molecular data and simulating a Hamiltonian
3. **The QITE Algorithm** - Quantum Imaginary Time Evolution for ground state preparation
4. **Scalable Resource Estimation** - Comparing QITE and QPE gate counts using `count_ops`

**Time:** about 45 minutes, with two small tasks marked **Your turn** (solutions in `qrisp_chemistry_tutorial_solutions.ipynb`). Part 1 is a warm-up: skip it if the lecture was enough. Part 2 is the core. Parts 3 and 4 show what the operators are used for.

**Finished early?** Try the stretch goal `double_slit_exercises.ipynb`: simulate the double-slit experiment with `QuantumFloat`s, automatic uncomputation, Grover's algorithm and the QFT.

In [ ]:
# Uncomment on Colab, Linux, macOS or WSL if the packages are not installed yet:
# %pip install qrisp pyscf matplotlib

import platform
if platform.system() == "Windows":
    print("Note: PySCF cannot be installed on native Windows.")
    print("  The tutorial falls back to a stored H2 Hamiltonian.")
    print("  For full functionality, run this notebook in WSL, Linux, macOS or on Colab.")


def check(test, hint, success="Correct!"):
    """Run test() and print ✅ on success or ❌ with a hint."""
    try:
        ok = bool(test())
    except NotImplementedError:
        ok, hint = False, "not implemented yet. " + hint
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}. " + hint
    print(f"✅ {success}" if ok else f"❌ Not yet: {hint}")

---
## 1. Qrisp Basics

### What is a QuantumVariable?

The central building block of every Qrisp program is the **QuantumVariable**. It abstracts away qubit management and lets you work with quantum data in a high-level, Pythonic way. Creating one is as simple as specifying the number of qubits:

In [ ]:
from qrisp import QuantumVariable

qv = QuantumVariable(3)
print(qv)

At this point the variable is in the $|000\rangle$ state. Printing it queries Qrisp's built-in simulator and shows us the measurement probabilities.

### Applying Gates

Qrisp provides familiar gate-application functions. You can apply a gate to a single qubit (by indexing) or to all qubits of a QuantumVariable at once:

In [ ]:
from qrisp import h, x, cx

# Put the first qubit into superposition
h(qv[0])

# Entangle qubits 0 and 1 with a CNOT
cx(qv[0], qv[1])

print(qv)

We created a Bell state across the first two qubits: $\frac{1}{\sqrt{2}}(|000\rangle + |110\rangle)$. The third qubit stays in $|0\rangle$.

You can also inspect the underlying circuit by printing the **QuantumSession**:

In [ ]:
print(qv.qs)

### Writing Quantum Functions

In Qrisp, quantum algorithms are written as regular Python functions that operate on QuantumVariables. This makes code reusable and composable:

In [ ]:
from qrisp import QuantumVariable, h, cx

def create_ghz(n):
    """Create an n-qubit GHZ state."""
    qv = QuantumVariable(n)
    h(qv[0])
    for i in range(n - 1):
        cx(qv[i], qv[i + 1])
    return qv

ghz = create_ghz(4)
print(ghz)

### Getting Measurements

There are several ways to extract results:

- **`print(qv)`** displays the measurement dictionary directly.
- **`qv.get_measurement()`** returns a dictionary mapping outcomes to probabilities.
- **`qv.qs.statevector()`** returns the full statevector (for small systems).

In [ ]:
results = ghz.get_measurement()
print("Measurement results:", results)

In [ ]:
sv = ghz.qs.statevector()
print("Statevector:")
sv

---
## 2. Simulating the $H_2$ Molecule

Now that we are comfortable with the basics, let's tackle a real-world problem: simulating the dynamics of the **Dihydrogen molecule** ($H_2$).

### Background: Second Quantization in a Nutshell

In quantum chemistry the behavior of electrons is modelled using **second quantization**. The key idea is that for each orbital $i$ there exist:

- A **creator** $a_i^\dagger$ that adds an electron to orbital $i$
- An **annihilator** $a_i$ that removes it

Electrons are **fermions**, so their ladder operators satisfy the anti-commutation relations:

$$\{a_i, a_j^\dagger\} = \delta_{ij}, \qquad \{a_i^\dagger, a_j^\dagger\} = \{a_i, a_j\} = 0$$

In particular, a direct consequence is the **Pauli exclusion principle**: $a_i^\dagger a_i^\dagger = 0$, meaning two electrons cannot occupy the same orbital.

Qrisp lets you build fermionic operators symbolically using the `c` (creator) and `a` (annihilator) functions:

In [ ]:
from qrisp.operators import c, a

O = a(0) * c(1) + a(1) * a(2)
print("Operator:", O)
print("Dagger:  ", O.dagger())

The `.reduce()` method simplifies expressions using anti-commutation laws. For instance $a_0 a_0 = 0$ (Pauli exclusion) and $a_1 a_2 - a_2 a_1 = -2 a_2 a_1$:

In [ ]:
O = a(0) * a(0) + a(1) * a(2) - a(2) * a(1)
print(O.reduce())

### Loading Molecular Data from PySCF

Qrisp provides a convenient interface to [PySCF](https://pyscf.org/) to load molecular Hamiltonians directly. The `FermionicOperator.from_pyscf` method computes the one- and two-electron integrals and returns a symbolic fermionic Hamiltonian.

> **Note:** PySCF is required for this section (`pip install pyscf`). On Windows, you may need to run this inside WSL. If PySCF is not available, the next cell will fall back to a hardcoded version of the $H_2$ Hamiltonian.

In [ ]:
from qrisp.operators import FermionicOperator, c, a

try:
    from pyscf import gto

    # Define the H2 molecule: two hydrogen atoms separated by 0.74 Angstrom
    mol = gto.M(atom="""H 0 0 0; H 0 0 0.74""", basis="sto-3g")
    H_ferm = FermionicOperator.from_pyscf(mol)
    print("Loaded H2 Hamiltonian from PySCF")

except ImportError:
    print("PySCF not available - using hardcoded H2 Hamiltonian (STO-3G, d=0.74 Å)")

    # Hardcoded fermionic Hamiltonian for H2 at 0.74 Angstrom in the STO-3G basis.
    # These coefficients were computed via PySCF and correspond to the one- and
    # two-electron integrals after mean-field (RHF) treatment.
    H_ferm = (
        -0.181210462015197 * a(0) * a(1) * c(2) * c(3)
        + 0.181210462015197 * a(0) * c(1) * c(2) * a(3)
        - 1.25330978664598  * c(0) * a(0)
        + 0.674755926814448 * c(0) * a(0) * c(1) * a(1)
        + 0.482500939335617 * c(0) * a(0) * c(2) * a(2)
        + 0.663711401350814 * c(0) * a(0) * c(3) * a(3)
        + 0.181210462015197 * c(0) * a(1) * a(2) * c(3)
        - 0.181210462015197 * c(0) * c(1) * a(2) * a(3)
        - 1.25330978664598  * c(1) * a(1)
        + 0.663711401350814 * c(1) * a(1) * c(2) * a(2)
        + 0.482500939335617 * c(1) * a(1) * c(3) * a(3)
        - 0.475068848772179 * c(2) * a(2)
        + 0.697651504490461 * c(2) * a(2) * c(3) * a(3)
        - 0.475068848772179 * c(3) * a(3)
    )

print(H_ferm)

### The Jordan-Wigner Embedding

To simulate fermionic operators on a qubit-based quantum computer, we need to map them to operators on qubits. The most established mapping is the **Jordan-Wigner transformation**:

$$a_k = A_k \prod_{i=0}^{k-1} Z_i$$

where $A_k = |0\rangle\langle 1|$. This transforms our fermionic Hamiltonian into a **QubitOperator**:

In [ ]:
H_qubit = H_ferm.to_qubit_operator()
print(H_qubit)

### QubitOperators vs FermionicOperators

The `QubitOperator` we just obtained lives in a different mathematical space than the `FermionicOperator` we started with:

| | **FermionicOperator** | **QubitOperator** |
|---|---|---|
| **Space** | Abstract fermion Fock space | Qubit space $(\mathbb{C}^2)^{\otimes n}$ |
| **Building blocks** | Creators $c_i^\dagger$ and annihilators $a_i$ | Pauli matrices $X, Y, Z$ and others ($A, C, P^0, P^1$) |
| **Algebra** | Anti-commutation relations | Tensor product of local operators |
| **Simulation** | Requires a mapping (e.g. Jordan-Wigner) | Can be simulated directly on qubits |

You can also build `QubitOperator`s directly from **Pauli strings** - this is useful when working with spin models or when you already have a qubit-level Hamiltonian:

In [ ]:
from qrisp.operators import X, Y, Z, A, C

# Build a simple Heisenberg-type QubitOperator from Pauli strings
H_heisenberg = X(0)*X(1)*X(2) + Y(0)*Y(1)*Y(2) + Z(0)*Z(1)*Z(2)
print("Hand-built QubitOperator:")
print(H_heisenberg)

# The operators A (annihilation) and C (creation) act on qubits
# but are simulated more efficiently than their Pauli decomposition
print("\nOperator with A/C terms:")
op = C(0)*A(1) + C(1)*A(0)
print(op)
print("\nSame operator in Pauli basis:")
print(op.to_pauli())

### Your turn: a transverse-field Ising chain (5 min)

Build the Hamiltonian of a 4-site transverse-field Ising chain from Pauli strings,

$$H_\text{Ising} = \sum_{i=0}^{2} Z_i Z_{i+1} + 0.5 \sum_{i=0}^{3} X_i,$$

and compute its exact ground-state energy with `ground_state_energy()`.

**Predict first:** is the ground-state energy above or below $-3$? (Without the $X$ terms, the alternating state $|0101\rangle$ has energy $-3$.)

*Your answer:* 

In [ ]:
H_ising = None  # TODO: build the operator
E_ising = None  # TODO: its ground-state energy
print("Ground-state energy:", E_ising)

In [ ]:
import numpy as np


def ising_reference(n=4, field=0.5):
    """Exact ground-state energy from explicit Kronecker products (independent of Qrisp)."""
    I2, Xm, Zm = np.eye(2), np.array([[0, 1], [1, 0]]), np.diag([1, -1])
    def op(single, site):
        mats = [single if k == site else I2 for k in range(n)]
        out = mats[0]
        for m in mats[1:]:
            out = np.kron(out, m)
        return out
    H = sum(op(Zm, i) @ op(Zm, i + 1) for i in range(n - 1)) + field * sum(op(Xm, i) for i in range(n))
    return np.linalg.eigvalsh(H)[0]


check(lambda: abs(E_ising - ising_reference()) < 1e-6,
      "Use sum(Z(i)*Z(i+1) for i in range(3)) + 0.5*sum(X(i) for i in range(4)), then .ground_state_energy().")

### Visualizing a Hamiltonian Simulation Circuit

Any `QubitOperator` can be turned into a quantum circuit via **Trotterization**. The `.trotterization()` method returns a function that, when applied to a `QuantumVariable`, appends the corresponding unitary $e^{-iHt}$ (approximated by Trotter steps) to the circuit. We can then inspect the circuit with `print(qv.qs)`.

Note that Qrisp does **not** decompose `A`/`C` operators into Pauli strings before simulation. Instead, it uses the much more efficient method of [Kornell and Selinger (2023)](https://arxiv.org/abs/2310.12256), which simulates terms like $A_i C_j + \text{h.c.}$ directly in a single circuit step. This is particularly beneficial for chemistry Hamiltonians obtained via Jordan-Wigner, which naturally contain many such terms.

In [ ]:
from qrisp import QuantumVariable

# Trotterize the hand-built operator from above (first order, 1 Trotter step, evolution time t = 1)
U_sim = H_heisenberg.trotterization()

# Create a fresh QuantumVariable and apply the simulation
qv = QuantumVariable(3)
U_sim(qv, t=1, steps=1)

# Print the resulting quantum circuit
print(qv.qs)

We can also compute the exact ground state energy classically (by diagonalizing the matrix representation) for reference:

In [ ]:
exact_energy = H_qubit.ground_state_energy()
print(f"Exact ground state energy: {exact_energy:.6f}")

### Performing Hamiltonian Simulation

Physical systems evolve under the Schrödinger equation:

$$i\hbar \frac{d}{dt}|\phi\rangle = H|\phi\rangle$$

**Hamiltonian simulation** means creating the state $|\phi, t\rangle = e^{-iHt}|\phi, 0\rangle$ on a quantum computer. Qrisp provides the `.trotterization()` method which returns a Python function that approximates this unitary using a Trotter product formula:

In [ ]:
from qrisp import QuantumVariable

U = H_ferm.trotterization()

def state_prep():
    # Create a 4-qubit register (4 spin-orbitals for H2 in STO-3G)
    electron_state = QuantumVariable(4)
    # Initialize a superposition: 2 electrons in lower orbitals + 1 electron in highest
    electron_state[:] = {"1100": 2**-0.5, "0001": 2**-0.5}
    # Evolve under the Hamiltonian for t=100 atomic time units with 20 Trotter steps
    U(electron_state, t=100, steps=20)
    return electron_state

### Evaluating a Physical Observable

After the simulation, we want to measure a physical quantity: the **particle number operator**

$$N = \sum_{i=0}^{3} a_i^\dagger a_i$$

This counts the number of electrons in the system. Since the Hamiltonian conserves particle number, we expect the expectation value to remain constant throughout the simulation.

To evaluate an expectation value in Qrisp, we use the `.expectation_value()` method. A key design choice is that this method does **not** take a state vector or a `QuantumVariable` as input - instead it takes a **state preparation function**. This function must create a fresh `QuantumVariable`, prepare the desired state, and return it. Qrisp then internally handles circuit construction, measurement grouping, and statistical evaluation. Besides being convenient from a compiler POV, this functional interface naturally reflects the fact that we can not clone quantum states. In order to measure the expectation value of an observable we therefore need knowledge of how to **prepare** the quantum state, not the quantum state itself.

In [ ]:
from qrisp.operators import c, a

N = sum(c(i) * a(i) for i in range(4))
expectation_value = N.expectation_value(state_prep, precision=0.01)()
print(f"Expected particle number: {expectation_value:.4f}")
print("(Should be ~1.5 because (2+1)/2 = 1.5)")

### Your turn: particle number of a superposition (5 min)

Write `state_prep_pair()`, which prepares $\tfrac{1}{\sqrt{2}}\left(|1100\rangle + |0011\rangle\right)$, and compute $\langle N\rangle$ for it with `expectation_value`.

**Predict first:** what is $\langle N\rangle$?

*Your answer:* 

In [ ]:
def state_prep_pair():
    qv = QuantumVariable(4)
    # TODO: prepare (|1100> + |0011>)/sqrt(2), as in state_prep above
    return qv


N_pair = None  # TODO: the expectation value of N for state_prep_pair
print("<N> =", N_pair)

In [ ]:
def pair_ok():
    meas = state_prep_pair().get_measurement()
    return set(meas) == {"1100", "0011"} and abs(N_pair - 2) < 0.05


check(pair_ok, "Use qv[:] = {'1100': 2**-0.5, '0011': 2**-0.5}, then N.expectation_value(state_prep_pair)().")

---
## 3. The QITE Algorithm

### What is Quantum Imaginary Time Evolution?

Finding the ground state of a Hamiltonian is one of the most important problems in quantum chemistry. **Quantum Imaginary Time Evolution (QITE)** is a powerful technique for this task.

The idea comes from the observation that in **classical** statistical mechanics, the operator $e^{-\beta H}$ (imaginary time evolution) projects onto the ground state as $\beta \to \infty$. However, $e^{-\beta H}$ is **not unitary**, so it cannot be directly implemented on a quantum computer. The idea of realizing imaginary time evolution on a quantum computer was first proposed by [Motta et al. (2019)](https://arxiv.org/abs/1901.07653).

The [Double-Bracket QITE (DB-QITE)](https://arxiv.org/abs/2412.04554) algorithm circumvents this by constructing a sequence of **unitary** approximations $U_k$ that converge towards the ground state. The recursive formula (Group Commutator approximation) is:

$$U_{k+1} = e^{i\sqrt{s_k}H} \; e^{i\sqrt{s_k}\omega_k} \; e^{-i\sqrt{s_k}H} \; U_k$$

where $e^{it\omega_k} = U_k e^{it|0\rangle\langle 0|} U_k^\dagger$ is a reflection around the current state $|\omega_k\rangle = U_k|0\rangle$.

In Qrisp, QITE is implemented as a function that takes:

| Parameter | Description |
|-----------|-------------|
| `qarg` | The quantum value to evolve |
| `U_0` | A function preparing the initial state |
| `exp_H` | A function performing forward Hamiltonian evolution $e^{-itH}$ |
| `s` | A list of evolution times for each step |
| `k` | The number of QITE steps |

### Applying QITE to $H_2$

Let's now apply QITE to the $H_2$ Hamiltonian we loaded in Section 2. We already have the qubit Hamiltonian `H_qubit` and know its exact ground state energy. Here we set QITE up for $H_2$. Simulating it needs many shots per step, so in Part 4 we count its gates instead.

In [ ]:
from qrisp import QuantumVariable, x

# H2 in STO-3G has 4 spin-orbitals = 4 qubits
n_qubits = 4

# Initial state: Hartree-Fock |1100⟩ (2 electrons in the lowest spin-orbitals)
def U_0(qv):
    x(qv[0])
    x(qv[1])

# Forward evolution using Trotterization of the H2 qubit Hamiltonian
U_trot = H_qubit.trotterization()
def exp_H(qv, t):
    U_trot(qv, t, steps=2)

print("Evolution functions defined.")

### Preparing the Initial State

A natural starting point is the **Hartree-Fock state** $|1100\rangle$, which places two electrons into the two lowest-energy spin-orbitals. This is the classical mean-field approximation to the ground state and provides a reasonable initial energy. Let's verify:

In [ ]:
def state_prep_init():
    qv = QuantumVariable(n_qubits)
    U_0(qv)
    return qv

E_0 = H_qubit.expectation_value(state_prep_init)()
print(f"Initial state energy (Hartree-Fock): {E_0:.6f}")
print(f"Exact ground state energy:           {exact_energy:.6f}")
print(f"Gap to close:                        {E_0 - exact_energy:.6f}")

---
## 4. Scalable Resource Estimation with `count_ops`

### The Problem of Scale

For small systems like $H_2$ we can directly simulate quantum circuits. But for **practically relevant** algorithms (hundreds or thousands of qubits), we can't simulate the circuit anymore - we can only **count** the gates.

Qrisp's Jasp infrastructure provides the `count_ops` decorator, which compiles a quantum function into a classical function that efficiently counts the number of each gate type - without ever building or simulating the full circuit. This works even for circuits with **billions of gates**, taking only milliseconds.

### Comparing QITE vs QPE for $H_2$

Let's compare two approaches for ground state energy estimation applied to our $H_2$ molecule:

- **QITE**: Quantum Imaginary Time Evolution (the algorithm we set up in Section 3)
- **QPE**: Quantum Phase Estimation, a textbook algorithm that estimates eigenvalues of a unitary

These two algorithms work very different, so it is not straightforward to draw clear conclusions from this comparison but it is still instructive to learn about the resource estimation feature.

We reuse the same `H_qubit`, `U_0`, and `exp_H` from the previous sections - the *exact same code* now powers resource estimation.

In [ ]:
from qrisp import QuantumVariable, QPE, count_ops, measure
from qrisp.qite import QITE
import jax.numpy as jnp

# We reuse H_qubit, U_0, exp_H, and n_qubits from Sections 2 & 3

Now let's define the QITE and QPE programs as `count_ops`-decorated functions. Note how concise this is - the same code that ran on the simulator is now being used for resource estimation. Even though $H_2$ is small (4 qubits), the same workflow applies seamlessly to molecules with hundreds of qubits.

In [ ]:
# Resource estimation for QITE (3 steps) on H2
@count_ops(meas_behavior="1")
def qite_resources():
    qv = QuantumVariable(n_qubits)
    s = jnp.array([0.1, 0.1, 0.1])
    QITE(qv, U_0, exp_H, s, 3)
    return measure(qv)

print("QITE gate counts (H2, 3 QITE steps):")
qite_ops = qite_resources()
print(qite_ops)

In [ ]:
# Resource estimation for QPE on H2
@count_ops(meas_behavior="1")
def qpe_resources(precision):
    qv = QuantumVariable(n_qubits)
    U_0(qv)
    
    def U_step(qv):
        exp_H(qv, 1.0)
    
    res = QPE(qv, U_step, precision=precision)
    return measure(res)

print("QPE gate counts (H2, 5-bit precision):")
qpe_ops = qpe_resources(5)
print(qpe_ops)
print("QPE gate counts (H2, 20-bit precision):")
qpe_ops = qpe_resources(20)
print(qpe_ops)

### Why This Matters

The key takeaway is that `count_ops` enables **resource estimation at practically relevant scales** without ever constructing the full circuit. We demonstrated it here on $H_2$ (4 qubits), but the same workflow applies identically to larger molecules: the only change is loading a different Hamiltonian. This is critical for:

- **Algorithm comparison**: Quickly determine which algorithm is more efficient for a given molecule
- **Hardware planning**: Estimate whether an algorithm fits within the coherence budget of a given quantum processor
- **Research**: Explore algorithmic tradeoffs without being limited by simulator memory

Under the hood, this is powered by **Jasp** - Qrisp's Jax-based compilation infrastructure that traces quantum programs into an intermediate representation and evaluates the gate counts purely classically. Even circuits with hundreds of millions of gates can be profiled in under a second.

---
## Summary

In this tutorial we:

1. Learned the basics of Qrisp: `QuantumVariable`, gate application, quantum functions, and measurements.
2. Loaded the $H_2$ molecular Hamiltonian from PySCF, embedded it via Jordan-Wigner, and performed Hamiltonian simulation using Trotterization.
3. Set up the **DB-QITE** algorithm for $H_2$, starting from the Hartree-Fock state.
4. Used `count_ops` to perform **scalable resource estimation**, comparing QITE and QPE gate counts for the same $H_2$ molecule - a workflow that scales seamlessly to larger molecular systems.

Qrisp's combination of a high-level programming model with Jasp-powered scalability makes it a practical tool for both quantum algorithm development and large-scale resource estimation.

---

<div style="background: linear-gradient(135deg, #5865F2 0%, #7289DA 100%); border-radius: 16px; padding: 40px; text-align: center; margin: 30px 0;">
<h1 style="color: white; margin: 0 0 15px 0; font-size: 2em;">🚀 Join the Qrisp Community!</h1>
<p style="color: #E8E8E8; font-size: 1.2em; margin: 0 0 25px 0;">
Got questions? Want to discuss quantum algorithms? Building something cool?<br>
Come hang out with us on Discord - we'd love to hear from you!
</p>
<a href="https://discord.gg/v5np7DeBaq" style="display: inline-block; background: white; color: #5865F2; font-size: 1.4em; font-weight: bold; padding: 15px 40px; border-radius: 10px; text-decoration: none;">
💬 Join our Discord
</a>
</div>